# Pipeline de Produção — Selagem FB17

Notebook orquestrador agendável como Seeq job.  
Executa a cada 1 hora: extrai dados, atualiza datasets analíticos, avalia gatilhos e grava no SharePoint.

**Fluxo**
1. Generators: `sku_dates` (Seeq) + `00_hour_prev` → `01_vida_rul` → `02_sinais_forca` → `03_padroes`
2. SKU normalization: adiciona `Media_norm` ao sinal de força
3. Avaliação de gatilhos v4.0: RISCO / CRITICO (AVISO / CONFIRMADO / FIM_DE_VIDA)
4. Gravação SharePoint (`ESCREVER_SP = True`)
5. Kairos Snapshot → lista `Kairos_Snapshots` (project-kairos)

In [ ]:
import sys
from pathlib import Path

# notebooks/ para mock seeq em dev local; fb17-datalab/ para imports src.*
_NB_DIR = Path.cwd()
_ROOT   = _NB_DIR.parent
for _p in (_NB_DIR, _ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from seeq import spy
spy.jobs.schedule("every 1 hour")

In [ ]:
# ── Configuração ──────────────────────────────────────────────────────────────
from src.connector import load_maquina, load_config

MAQUINA     = load_maquina()   # lê project.maquina do config.yaml
cfg         = load_config()    # config completo — usado por Kairos Snapshot
LIST_NAME   = "Gatilhos_Selagem"
SP_URL      = ""    # defina SP_URL=https://... em sharepoint.ev
ESCREVER_SP = True  # alterar para False em dry-run

PATH_HOUR_PREV = _NB_DIR / "00_hour_prev.csv"
PATH_VIDA_RUL  = _NB_DIR / "01_vida_rul.csv"
PATH_WEIBULL   = _NB_DIR / "01_weibull_params.json"
PATH_SINAIS    = _NB_DIR / "02_sinais_forca.csv"
PATH_PADROES   = _NB_DIR / "03_padroes.csv"
PATH_SKU_DATES = _NB_DIR / "sku_dates.csv"
PATH_TROCA     = _NB_DIR / "troca_modulo.csv"
PATH_STATE     = _NB_DIR / f"state_{MAQUINA.lower()}.json"

## Etapa 1 — Generators

Atualiza os quatro datasets analíticos a partir dos dados mais recentes do Seeq.

In [ ]:
from src.generators.gen_sku_dates  import run as gen_sku_dates
from src.generators.gen_hour_prev  import run as gen_hour_prev
from src.generators.gen_vida_rul   import run as gen_vida_rul
from src.generators.gen_sinais     import run as gen_sinais
from src.generators.gen_padroes    import run as gen_padroes

print("[0/4] Atualizando Phantom dates via Seeq...")
df_sku_raw = gen_sku_dates(output_path=PATH_SKU_DATES)
n_phantom = df_sku_raw["phantom"].notna().sum()
print(f"      sku_dates: {len(df_sku_raw):,} leituras | {n_phantom:,} com Phantom Code")

print("[1/4] Extraindo dados do Seeq...")
df_hour = gen_hour_prev(output_path=PATH_HOUR_PREV, troca_csv=PATH_TROCA)
print(f"      00_hour_prev: {len(df_hour):,} leituras")

print("[2/4] Calculando vida e Weibull...")
df_vida = gen_vida_rul(
    input_path=PATH_HOUR_PREV,
    output_csv=PATH_VIDA_RUL,
    output_json=PATH_WEIBULL,
    troca_csv=PATH_TROCA,
)
n_ciclos = df_vida["ciclo_id"].nunique() if "ciclo_id" in df_vida.columns else "?"
print(f"      01_vida_rul: {len(df_vida):,} linhas | {n_ciclos} ciclos")

print("[3/4] Calculando features de força...")
df_sinais = gen_sinais(
    input_path=PATH_HOUR_PREV,
    output_path=PATH_SINAIS,
    troca_csv=PATH_TROCA,
)
print(f"      02_sinais_forca: {len(df_sinais):,} linhas")

print("[4/4] Calculando padrões de detecção...")
df_padroes = gen_padroes(
    input_vida_rul=PATH_VIDA_RUL,
    input_sinais=PATH_SINAIS,
    output_path=PATH_PADROES,
    troca_csv=PATH_TROCA,
)
n_gen = df_padroes["genuino"].sum() if "genuino" in df_padroes.columns else "?"
print(f"      03_padroes: {len(df_padroes)} ciclos ({n_gen} genuínos)")

## Etapa 2 — Normalização por Phantom

Carrega `02_sinais_forca.csv` já com `Media_norm` gerada por phantom auto-calibrado.  
Elimina o Paradoxo de Simpson sem catálogo estático — fator calibrado pelos primeiros 10 dias de cada ciclo.

In [ ]:
import pandas as pd

# 02_sinais_forca.csv já tem Media_norm gerada por gen_sinais (phantom auto-calibrado)
df_hourly = pd.read_csv(PATH_SINAIS, parse_dates=["Timestamp"])
df_hourly["Timestamp"] = pd.to_datetime(df_hourly["Timestamp"], utc=True)
df_hourly = df_hourly.set_index("Timestamp").sort_index()

n_norm    = df_hourly["Media_norm"].notna().sum() if "Media_norm" in df_hourly.columns else 0
n_total   = len(df_hourly)
cobertura = n_norm / n_total if n_total > 0 else 0.0
print(f"Media_norm: {n_norm:,}/{n_total:,} leituras ({cobertura:.0%})")

if n_norm > 0 and "phantom_fator" in df_hourly.columns:
    fator_medio  = df_hourly["phantom_fator"].mean()
    moda_phantom = df_hourly["phantom_codigo"].mode()
    phantom_dom  = moda_phantom.iloc[0] if not moda_phantom.empty else None
    print(f"Fator phantom médio : {fator_medio:.3f}")
    if phantom_dom is not None:
        print(f"Phantom dominante   : {phantom_dom}")

# Usa Media_norm se cobertura >= 50%; caso contrário, fallback para força bruta
media_col = "Media_norm" if cobertura >= 0.50 else "Media"
print(f"Coluna p/ trigger: '{media_col}'")

## Etapa 3 — Estado do ciclo atual

In [ ]:
from src.predictor import load_troca_dates

troca_dates  = load_troca_dates(PATH_TROCA)
ultima_troca = troca_dates[-1]
hoje         = df_hourly.index[-1].normalize()
idade_dias   = (hoje - ultima_troca).days

print(f"Última troca confirmada : {ultima_troca.date()}")
print(f"Última leitura          : {hoje.date()}")
print(f"Idade do rolo           : {idade_dias} dias")

## Etapa 4 — Avaliação de gatilhos

In [ ]:
import os

sp_client = None
if ESCREVER_SP:
    try:
        from dotenv import dotenv_values
        # sharepoint.ev fica na raiz do projeto (fb17-datalab/), não em notebooks/
        _ev = dotenv_values(_ROOT / "sharepoint.ev")
        _sp_url  = _ev.get("SP_URL") or "https://kimberlyclark.sharepoint.com/Sites/H945"
        _sp_user = _ev.get("SP_USER")
        _sp_pass = _ev.get("SP_PASS")
        from src.sharepoint_methods import SharePointClient
        sp_client = SharePointClient(url=_sp_url, username=_sp_user, password=_sp_pass)
        print(f"SharePoint: conectado ({_sp_url})")
    except Exception as _e:
        print(f"SharePoint: falha na conexão ({_e}) — modo dry-run")
else:
    print("SharePoint: dry-run (ESCREVER_SP=False)")

In [ ]:
from src.trigger_engine import TriggerEngine

engine = TriggerEngine(maquina=MAQUINA, state_path=PATH_STATE)

print(f"Avaliando {hoje.date()} | rolo com {idade_dias} dias | sinal='{media_col}'")
events = engine.evaluate(
    df_hourly=df_hourly,
    troca_date=ultima_troca,
    sp_client=sp_client,
    list_name=LIST_NAME,
    media_col=media_col,
    troca_dates=troca_dates,
)

print()
if events:
    print(f"{len(events)} disparo(s):")
    for ev in events:
        sp_tag = f" [SP ID={ev.sp_item_id}]" if ev.sp_item_id else ""
        print(f"  [{ev.gatilho}] {ev.severidade}{sp_tag}")
        print(f"  {ev.mensagem.splitlines()[0]}")
else:
    print("Nenhum gatilho disparado.")

In [ ]:
# ── Kairos Snapshot + Efêmérides Kepler ───────────────────────────
# Envia estado atual ao SharePoint List Kairos_Snapshots (Kairos central)
import math

def _nivel_from_snap(snap, cfg):
    p = snap.get("p_risk", 0)
    if p >= cfg["trigger"]["limiar_p_risk"]: return "CRITICO"
    if p >= cfg["trigger"]["aviso_p_risk"]:  return "RISCO"
    return "NOMINAL"

def _severidade_from_snap(snap, cfg):
    p = snap.get("p_risk", 0)
    if p >= cfg["trigger"]["limiar_p_risk"]: return "CRITICA"
    if p >= cfg["trigger"]["aviso_p_risk"]:  return "ALTA"
    return "NOMINAL"

def _janela_label(p_risk):
    if p_risk >= 0.70: return "CRITICA"
    if p_risk >= 0.40: return "PLANEJADA"
    return "MONITORAR"

def _acao_recomendada(snap, cfg):
    p    = snap.get("p_risk", 0)
    dias = max(0.0, (cfg["trigger"]["weibull_eta_h"] / 24) - snap.get("age_days", 0))
    if p >= cfg["trigger"]["limiar_p_risk"]:
        return f"Substituicao urgente - p_risk={p:.2f}, dias restantes={dias:.0f}"
    if p >= cfg["trigger"]["aviso_p_risk"]:
        return f"Planejar substituicao - p_risk={p:.2f}, dias restantes={dias:.0f}"
    return f"Monitorar - p_risk={p:.2f}, dias restantes={dias:.0f}"

if ESCREVER_SP:
    try:
        from src.connector import load_config
        from src.trigger_engine import compute_p_risk_snapshot
        cfg = load_config(_ROOT / "config.yaml")

        snap_k   = compute_p_risk_snapshot(df_hourly, ultima_troca, hoje)
        eta_dias = cfg["trigger"]["weibull_eta_h"] / 24

        snapshot_dict = {
            "fonte":             "fb17-datalab",
            "maquina":           MAQUINA,
            "parte_mecanica":    "maintacker",
            "timestamp_iso":     snap_k.get("today", pd.Timestamp.now()).isoformat(),
            "idade_dias":        float(snap_k.get("age_days", 0)),
            "eta_ajustado_dias": eta_dias,
            "dias_restantes":    max(0.0, eta_dias - float(snap_k.get("age_days", 0))),
            "p_risk":            float(snap_k.get("p_risk", 0)),
            "age_risk":          float(snap_k.get("age_risk", 0)),
            "signal_score":      float(snap_k.get("sig_score", snap_k.get("signal_score", 0))),
            "nivel":             _nivel_from_snap(snap_k, cfg),
            "severidade":        _severidade_from_snap(snap_k, cfg),
            "mean_3d":           snap_k.get("mean_3d"),
            "slope_7d":          snap_k.get("slope_7d"),
            "proj_48h":          snap_k.get("proj_48h"),
            "urgencia_dias":     max(0.0, eta_dias - float(snap_k.get("age_days", 0))),
            "janela_label":      _janela_label(float(snap_k.get("p_risk", 0))),
            "acao_recomendada":  _acao_recomendada(snap_k, cfg),
        }

        # ── Efêmérides Kepler (M, E, e) ───────────────────────────────────────────
        # M = age_risk x 2pi  (Anomalia Media: previsao Weibull)
        # E = p_risk   x 2pi  (Anomalia Excentrica: risco hibrido observado)
        # e = (E - M) / sin(E)  [equacao de Kepler invertida: M = E - e*sin(E)]
        _M_rad = float(snap_k.get("age_risk", 0)) * 2 * math.pi
        _E_rad = float(snap_k.get("p_risk",   0)) * 2 * math.pi
        _sin_E = math.sin(_E_rad)
        _e_kep = min((_E_rad - _M_rad) / _sin_E, 0.99) if abs(_sin_E) > 1e-6 else 0.0
        _e_kep = max(_e_kep, 0.0)

        _sinal_atual = float(snap_k.get("mean_3d") or 0)
        _vel_sinal   = float(snap_k.get("slope_7d") or 0)  # N/dia
        _limite_inf  = float(
            cfg["trigger"].get("limiar_forca",
            cfg["trigger"].get("risco_min_forca", 800))
        )
        _dias_ate = (
            round((_sinal_atual - _limite_inf) / abs(_vel_sinal), 1)
            if _vel_sinal < 0 and _sinal_atual > _limite_inf else None
        )

        snapshot_dict.update({
            "anomalia_media":      round(float(snap_k.get("age_risk", 0)), 6),
            "eccentricidade":      round(_e_kep, 6),
            "anomalia_excentrica": round(float(snap_k.get("p_risk", 0)), 6),
            "velocidade_sinal":    round(_vel_sinal, 4),
            "sinal_atual":         round(_sinal_atual, 2),
            "limite_inferior":     _limite_inf,
            "dias_ate_limite":     _dias_ate,
            "unidade_sinal":       "N",
        })

        _ids    = sp_client.insert_list_item("Kairos_Snapshots", [snapshot_dict])
        item_id = _ids[0] if _ids else None
        print(f"[kairos] Snapshot -> Kairos_Snapshots ID={item_id}")
    except Exception as _e_kairos:
        print(f"[kairos] Aviso: snapshot nao enviado - {_e_kairos}")
else:
    print("[kairos] ESCREVER_SP=False - snapshot nao enviado (dry-run)")


## Diagnóstico (execução manual)

Snapshot completo dos indicadores calculados para hoje. Útil para debug e auditoria.

In [ ]:
from src.trigger_engine import compute_p_risk_snapshot

snap = compute_p_risk_snapshot(df_hourly, ultima_troca, hoje)
print("Indicadores calculados:")
for k, v in snap.items():
    print(f"  {k:<22}: {v}")